# Temporal memory

The default reduced profile now regenerates every quantitative panel from genuine sample-level trials. Sequential MNIST uses genuine MNIST images; N-MNIST, SHD, and DVS Gesture use small deterministic fixtures drawn from their official test distributions; retention and capability use the repository's `TemporalMSNN`/`TemporalMCSNN`; store–recall uses live `DeviceLeaky` and `AdaptiveLeaky` trials. A provenance-complete full-sweep seed archive remains an explicit optional replay.

## Configuration, reduced fixtures, and optional archive

Reduced scores are validation results on documented nonstandard small splits, not replacements for manuscript benchmark scores. Set `MNN_USE_TEMPORAL_ARCHIVE=1` to replay a compatible `data/results/temporal_seed_archive.json`; malformed, aggregate-only, repeated-mean, or synthetic-CI archives are rejected. Saving remains optional and disabled by default.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import hashlib, json, os, shutil
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch, Rectangle

ROOT = Path.cwd().resolve()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
RUN_PROFILE = os.getenv("MNN_RUN_PROFILE", "reduced")  # reduced | publication | smoke
DEVICE = os.getenv("MNN_DEVICE", "auto")
WORKERS = os.getenv("MNN_WORKERS", "auto")
SAVE_FIGURES = os.getenv("MNN_SAVE_FIGURES", "0") == "1"
OUTPUT_DIR = Path(os.getenv("MNN_OUTPUT_DIR", str(ROOT / "experiments" / "generated_figures")))
OVERWRITE = os.getenv("MNN_OVERWRITE", "0") == "1"
RUN_EXTERNAL_DATA = os.getenv("MNN_RUN_EXTERNAL_DATA", "0") == "1"
ALLOW_DATA_DOWNLOADS = os.getenv("MNN_ALLOW_DATA_DOWNLOADS", "0") == "1"
USE_TEMPORAL_ARCHIVE = os.getenv("MNN_USE_TEMPORAL_ARCHIVE", "0") == "1"
assert RUN_PROFILE in {"reduced", "publication", "smoke"}

try:
    import torch
except ImportError:
    torch = None
if RUN_PROFILE in {"reduced", "smoke"} and torch is None:
    raise RuntimeError("The reduced and smoke profiles require the repository's Torch environment.")
if DEVICE == "auto" and torch is not None:
    DEVICE_RESOLVED = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
else:
    DEVICE_RESOLVED = DEVICE if DEVICE != "auto" else "cpu"

MANIFEST = json.loads((ROOT / "experiments" / "figure_manifest.json").read_text())
SPEC = {row["filename"]: row for row in MANIFEST["figures"]}
FIGURE_REPORT = []

def _hash(value):
    if value is None:
        return None
    if isinstance(value, Path):
        return hashlib.sha256(value.read_bytes()).hexdigest() if value.exists() else None
    payload = json.dumps(value, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(payload).hexdigest()

def _guard_save(name, provenance):
    if provenance in {"placeholder", "reference-only", "external-gated"}:
        raise RuntimeError(f"Refusing to save non-claimable figure: {name}")
    if "manuscript" in str(OUTPUT_DIR).lower() and not OVERWRITE:
        raise RuntimeError("Writing into a manuscript directory requires MNN_OVERWRITE=1")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    target = OUTPUT_DIR / name
    if target.exists() and not OVERWRITE:
        raise FileExistsError(f"Refusing to overwrite {target}")
    return target

def finish(fig, name, data_source=None, reference=None, seeds=(), *, dpi=220, bbox_inches="tight", pad_inches=None,
           provenance_class=None, claim_status=None):
    spec = dict(SPEC[name])
    if provenance_class is not None:
        spec["provenance_class"] = provenance_class
    if claim_status is not None:
        spec["claim_status"] = claim_status
    display(fig)
    saved = None
    if SAVE_FIGURES:
        saved = _guard_save(name, spec["provenance_class"])
        save_kw = dict(dpi=dpi, facecolor="white")
        if bbox_inches is not None: save_kw["bbox_inches"] = bbox_inches
        if pad_inches is not None: save_kw["pad_inches"] = pad_inches
        fig.savefig(saved, **save_kw)
    FIGURE_REPORT.append({
        **spec, "profile": RUN_PROFILE, "device": DEVICE_RESOLVED,
        "seeds": list(seeds), "data_sha256": _hash(data_source),
        "reference_sha256": _hash(reference), "saved_path": str(saved) if saved else None,
    })
    plt.close(fig)
    return None

def finish_asset(name, source):
    source = Path(source)
    display(Image(filename=str(source)))
    saved = None
    if SAVE_FIGURES:
        saved = _guard_save(name, SPEC[name]["provenance_class"])
        shutil.copyfile(source, saved)
    FIGURE_REPORT.append({**SPEC[name], "profile": RUN_PROFILE, "device": DEVICE_RESOLVED,
        "seeds": [], "data_sha256": _hash(source), "reference_sha256": _hash(source),
        "saved_path": str(saved) if saved else None})


## Streamed architectures

The two diagrams distinguish feature-stream integration from convolutional event-frame integration while keeping the temporal state explicit.

In [ ]:
NOTEBOOK_NAME = "03_temporal_memory.ipynb"
TEAL = "#3aa07a"; BLUE = "#2f4b8f"; GREY = "#9aa6b2"; AMBER = "#e0a93b"
RED = "#c0392b"; INK = "#2b2b2b"; MEMBG = "#eaf0fb"; LIFBG = "#eaf5ef"

def neuron_col(ax, x, ys, r, fc, ec):
    for y in ys: ax.add_patch(Circle((x, y), r, fc=fc, ec=ec, lw=1.2, zorder=3))

def connect(ax, x0, ys0, x1, ys1, color, lw=0.5, alpha=0.5):
    for y0 in ys0:
        for y1 in ys1: ax.plot([x0, x1], [y0, y1], color=color, lw=lw, alpha=alpha, zorder=1)

def crossbar(ax, x, y, w, h, n=4, color=BLUE):
    xs = np.linspace(x + 0.12*w, x + 0.88*w, n); ys = np.linspace(y + 0.15*h, y + 0.85*h, n)
    for yy in ys: ax.plot([x + 0.06*w, x + 0.94*w], [yy, yy], color=GREY, lw=0.9, zorder=2)
    for xx in xs: ax.plot([xx, xx], [y + 0.06*h, y + 0.94*h], color=GREY, lw=0.9, zorder=2)
    for xx in xs:
        for yy in ys: ax.add_patch(Circle((xx, yy), 0.012, fc=color, ec="none", zorder=3))

def membrane_glyph(ax, x, y, w, h, color=TEAL):
    t = np.linspace(0, 1, 120); rise = 1 - np.exp(-t / 0.06); yv = rise * np.exp(-t / 0.5); yv /= yv.max()
    ax.plot(x + 0.1*w + t*0.8*w, y + 0.2*h + yv*0.6*h, color=color, lw=2.0, zorder=3)

fig = plt.figure(figsize=(10, 7.2))
axA = fig.add_axes([0.02, 0.53, 0.96, 0.42]); axA.axis("off")
axB = fig.add_axes([0.02, 0.03, 0.96, 0.42]); axB.axis("off")
for ax in (axA, axB): ax.set_ylim(0.15, 4)
axA.set_xlim(0.1, 8.4); axB.set_xlim(0.1, 9.4)

# (a) TemporalMSNN -- canonical generator geometry and labels.
axA.text(0.0, 3.9, "(a) TemporalMSNN --- dense streamed network", fontsize=13.5, color=INK, fontweight="bold")
yc = 2.0; axA.text(1.05, 3.35, "streamed input  $x[1],\\,x[2],\\dots,x[T]$", fontsize=12, color=INK, ha="center")
for k in range(3): axA.add_patch(FancyBboxPatch((0.55+k*0.16, 1.2), 0.28, 1.5, boxstyle="round,pad=0.01", fc="#f4f6f9", ec=GREY, lw=1.1, zorder=2))
in_ys = np.linspace(1.4, 2.6, 4); neuron_col(axA, 1.05, in_ys, 0.10, "white", GREY)
axA.add_patch(FancyBboxPatch((2.1, 1.05), 1.7, 1.9, boxstyle="round,pad=0.02", fc=MEMBG, ec=BLUE, lw=1.4, zorder=1)); crossbar(axA, 2.1, 1.05, 1.7, 1.9)
axA.text(2.95, 0.82, "memristive crossbar\n(Poole--Frenkel synapse)", fontsize=11.5, color=BLUE, ha="center", va="top"); connect(axA, 1.15, in_ys, 2.2, np.linspace(1.4,2.6,4), GREY)
hid_ys = np.linspace(1.35, 2.65, 5); axA.add_patch(FancyBboxPatch((4.25,1.0),1.9,2.0,boxstyle="round,pad=0.02",fc=LIFBG,ec=TEAL,lw=1.4,zorder=1))
neuron_col(axA,4.75,hid_ys,0.11,"white",TEAL); membrane_glyph(axA,5.15,1.35,0.9,1.3); axA.text(5.2,3.05,"DeviceLeaky LIF,  $\\beta=e^{-\\Delta t/\\tau_{\\rm leak}}$",fontsize=11.5,color=TEAL,ha="center",va="bottom")
connect(axA,3.75,np.linspace(1.4,2.6,4),4.65,hid_ys,GREY,alpha=0.35); out_ys=np.linspace(1.7,2.3,2); neuron_col(axA,7.1,out_ys,0.12,"white",INK); connect(axA,4.85,hid_ys,7.0,out_ys,GREY,alpha=0.35); axA.text(7.1,1.35,"readout",fontsize=11.5,color=INK,ha="center")
for p0,p1,style in [((5.6,1.0),(5.6,0.6),"-"),((5.6,0.6),(4.6,0.6),"-"),((4.6,0.6),(4.6,1.0),"-|>")]: axA.add_patch(FancyArrowPatch(p0,p1,arrowstyle=style,color=TEAL,lw=1.4,mutation_scale=12,zorder=4))
axA.text(5.1,0.28,"membrane state carried across $t$",fontsize=11,color=TEAL,ha="center",va="center",style="italic")
for x0,x1 in [(1.25,2.05),(3.85,4.2),(6.2,6.95)]: axA.add_patch(FancyArrowPatch((x0,yc),(x1,yc),arrowstyle="-|>",color=INK,lw=1.3,mutation_scale=13,zorder=5))

# (b) TemporalMCSNN -- canonical generator geometry and labels.
axB.text(0.0,3.9,"(b) TemporalMCSNN --- convolutional streamed network",fontsize=13.5,color=INK,fontweight="bold"); axB.text(1.0,3.35,"streamed clip  (frames)",fontsize=12,color=INK,ha="center")
for k,off in enumerate([0.0,0.13,0.26]): axB.add_patch(Rectangle((0.45+off,1.35+off),0.85,0.85,fc="#f4f6f9",ec=GREY,lw=1.1,zorder=2+k))
axB.text(1.0,1.1,"$C\\times H\\times W$",fontsize=11,color=INK,ha="center",va="top")
def fmap_stack(ax,x,y,n,w,h,color):
    for k in range(n): ax.add_patch(Rectangle((x+k*0.10,y+k*0.10),w,h,fc=MEMBG,ec=color,lw=1.1,zorder=2+k))
fmap_stack(axB,2.2,1.4,3,0.7,0.7,BLUE); fmap_stack(axB,3.3,1.5,3,0.55,0.55,BLUE); axB.text(3.0,0.95,"memristive conv $\\times 2$\n(Poole--Frenkel)",fontsize=11.5,color=BLUE,ha="center",va="top")
hid_ys=np.linspace(1.4,2.6,5); axB.add_patch(FancyBboxPatch((4.9,1.0),1.9,2.0,boxstyle="round,pad=0.02",fc=LIFBG,ec=TEAL,lw=1.4,zorder=1)); neuron_col(axB,5.4,hid_ys,0.11,"white",TEAL); membrane_glyph(axB,5.8,1.35,0.9,1.3)
axB.text(5.85,3.05,"DeviceLeaky LIF,  $\\beta=e^{-\\Delta t/\\tau_{\\rm leak}}$",fontsize=11.5,color=TEAL,ha="center",va="bottom"); out_ys=np.linspace(1.7,2.3,2); neuron_col(axB,7.8,out_ys,0.12,"white",INK); connect(axB,5.5,hid_ys,7.7,out_ys,GREY,alpha=0.35); axB.text(7.8,1.35,"readout",fontsize=11.5,color=INK,ha="center")
axB.add_patch(FancyArrowPatch((8.05,2.0),(9.0,2.0),arrowstyle="-|>",color=RED,lw=1.3,mutation_scale=12,ls=(0,(4,2)))); axB.text(9.02,1.55,"decision read\nafter delay $D$",fontsize=10.5,color=RED,ha="right")
for p0,p1,style in [((6.25,1.0),(6.25,0.6),"-"),((6.25,0.6),(5.25,0.6),"-"),((5.25,0.6),(5.25,1.0),"-|>")]: axB.add_patch(FancyArrowPatch(p0,p1,arrowstyle=style,color=TEAL,lw=1.4,mutation_scale=12,zorder=4))
axB.text(5.75,0.28,"membrane state carried across frames",fontsize=11,color=TEAL,ha="center",va="center",style="italic")
for x0,x1 in [(1.45,2.05),(4.0,4.85),(6.85,7.65)]: axB.add_patch(FancyArrowPatch((x0,2.0),(x1,2.0),arrowstyle="-|>",color=INK,lw=1.3,mutation_scale=13,zorder=5))
finish(fig, "fig_temporal_arch.png", {"generator": "canonical temporal architecture", "version": 1}, dpi=200)


## Live delay–retention law

For each seed, reduced sequential-MNIST and N-MNIST models are trained once with the longest retained state. The exact trained weights are then evaluated across device leak constants and blank delays by changing only the `DeviceLeaky` decay. This isolates the retention mechanism while avoiding a separate model fit for every plotted grid cell. The no-memory curve uses the same weights with an effectively instantaneous leak.

In [ ]:
SEED_ARCHIVE = ROOT / "data" / "results" / "temporal_seed_archive.json"
REFERENCE_DIR = ROOT / "experiments" / "assets" / "references"
TEMPORAL_FIXTURE = ROOT / "data" / "fixtures" / "temporal_reduced_datasets.npz"
MNIST_FIXTURE = ROOT / "data" / "fixtures" / "homeostasis_reduced_datasets.npz"
REQUIRED_RUNNER = "mnn-temporal-full-v1"


def boot_ci(v, n=5000, seed=0):
    v = np.asarray(v, float)
    if len(v) == 1:
        return np.array([v[0], v[0]])
    rng = np.random.default_rng(seed)
    b = v[rng.integers(0, len(v), size=(n, len(v)))].mean(axis=1)
    return np.percentile(b, [2.5, 97.5])


def _seed_rows(rows, width, label, nseeds):
    a = np.asarray(rows, float)
    if a.shape != (nseeds, width) or not np.isfinite(a).all():
        raise ValueError(f"invalid seed array for {label}: {a.shape}")
    if len(np.unique(a, axis=0)) == 1:
        raise ValueError(f"repeated aggregate mean rejected as seed evidence: {label}")
    return a


def load_seed_archive(path=SEED_ARCHIVE):
    if not path.exists():
        return None
    z = json.loads(path.read_text()); meta = z.get("metadata", {})
    required = {"runner", "runner_code_sha256", "created_utc", "profile", "seeds",
                "config", "dataset_sha256", "software"}
    if required - set(meta):
        raise ValueError(f"seed archive metadata missing: {sorted(required-set(meta))}")
    seeds = meta["seeds"]; n = len(seeds)
    hashes = [meta["runner_code_sha256"], *meta["dataset_sha256"].values()]
    if (meta["runner"] != REQUIRED_RUNNER or n < 2 or len(set(seeds)) != n or
            any(len(h) != 64 or any(c not in "0123456789abcdef" for c in h.lower())
                for h in hashes)):
        raise ValueError("unrecognised runner, hashes or seeds")
    if set(z["retention"]) != {"dense", "conv"}:
        raise ValueError("incomplete retention archive")
    if set(z["capability"]) != {"seq-MNIST", "N-MNIST", "SHD", "DVS128 Gesture"}:
        raise ValueError("incomplete capability archive")
    for modality in ("dense", "conv"):
        panel = z["retention"][modality]; width = len(panel["gaps"])
        for key, rows in panel["acc"].items():
            _seed_rows(rows, width, f"{modality}/{key}", n)
    if set(z["store_recall"]) != {"ideal", "device healthy", "device faulty"}:
        raise ValueError("incomplete store-recall archive")
    for rung, panel in z["store_recall"].items():
        for key, rows in panel["acc"].items():
            _seed_rows(rows, len(panel["delays"]), f"{rung}/{key}", n)
    for key, rows in z["capability"].items():
        a = np.asarray(rows, float)
        if a.shape != (n,) or not np.isfinite(a).all() or len(np.unique(a)) == 1:
            raise ValueError(f"invalid capability seeds: {key}")
    return z


def show_reference_only(name, reason):
    target = REFERENCE_DIR / name
    display(Image(filename=str(target)))
    FIGURE_REPORT.append({**SPEC[name], "profile": RUN_PROFILE,
        "device": DEVICE_RESOLVED, "seeds": [], "data_sha256": None,
        "reference_sha256": _hash(target), "saved_path": None,
        "provenance_class": "reference-only", "claim_status": "not-regenerated",
        "reason": reason})


def temporal_budget(profile):
    if profile == "smoke":
        return {"seeds": [0], "dense_epochs": 2, "event_epochs": 2, "mnist_train_per_class": 8,
                "mnist_test_per_class": 4, "event_train_per_class": 4,
                "event_test_per_class": 2, "batch": 20,
                "dense_gaps": [0, 5, 10], "conv_gaps": [0, 5, 10]}
    return {"seeds": [0, 1, 2], "dense_epochs": 15, "event_epochs": 25, "mnist_train_per_class": 50,
            "mnist_test_per_class": 25, "event_train_per_class": 12,
            "event_test_per_class": 8, "batch": 32,
            "dense_gaps": [0, 5, 10, 20, 40],
            "conv_gaps": [0, 5, 10, 20, 40]}


def _load_reduced_fixtures():
    if not TEMPORAL_FIXTURE.exists() or not MNIST_FIXTURE.exists():
        raise FileNotFoundError("Reduced temporal fixtures are missing")
    tz = np.load(TEMPORAL_FIXTURE, allow_pickle=False)
    mz = np.load(MNIST_FIXTURE, allow_pickle=False)
    metadata = {"temporal": json.loads(str(tz["metadata_json"].item())),
                "mnist": json.loads(str(mz["metadata_json"].item()))}
    return tz, mz, metadata


def _split_class_rows(x, y, train_per_class, test_per_class):
    train, test = [], []
    for label in sorted(np.unique(y)):
        idx = np.flatnonzero(y == label)
        if len(idx) < train_per_class + test_per_class:
            raise ValueError(f"insufficient reduced samples for class {label}")
        train.extend(idx[:train_per_class]); test.extend(idx[train_per_class:train_per_class+test_per_class])
    return x[train], y[train], x[test], y[test]


def _normalise_events(x):
    x = torch.as_tensor(x, dtype=torch.float32)
    return 10.0 * torch.log1p(x) / np.log(256.0)


def _task_data(task, tz, mz, budget):
    if task == "seq-MNIST":
        tr_x, tr_y, _, _ = _split_class_rows(
            mz["mnist_train_x"], mz["mnist_train_y"],
            budget["mnist_train_per_class"], 1)
        _, _, te_x, te_y = _split_class_rows(
            mz["mnist_test_x"], mz["mnist_test_y"], 1,
            budget["mnist_test_per_class"])
        return (torch.as_tensor(tr_x, dtype=torch.float32).div(255).squeeze(1), tr_y,
                torch.as_tensor(te_x, dtype=torch.float32).div(255).squeeze(1), te_y)
    key = {"N-MNIST": "nmnist", "SHD": "shd", "DVS128 Gesture": "dvs"}[task]
    tr_x, tr_y, te_x, te_y = _split_class_rows(
        tz[f"{key}_x"], tz[f"{key}_y"], budget["event_train_per_class"],
        budget["event_test_per_class"])
    return _normalise_events(tr_x), tr_y, _normalise_events(te_x), te_y


def _loaders(data, seed, batch):
    from torch.utils.data import DataLoader, TensorDataset
    tr_x, tr_y, te_x, te_y = data
    pin = DEVICE_RESOLVED == "cuda"
    train = DataLoader(TensorDataset(torch.as_tensor(tr_x), torch.as_tensor(tr_y, dtype=torch.long)),
                       batch_size=batch, shuffle=True,
                       generator=torch.Generator().manual_seed(seed), pin_memory=pin)
    test = DataLoader(TensorDataset(torch.as_tensor(te_x), torch.as_tensor(te_y, dtype=torch.long)),
                      batch_size=batch, shuffle=False, pin_memory=pin)
    return train, test


def _temporal_config(tau=30.0):
    return {"ideal": True, "homeostasis_dropout": False,
            "lif_tau_leak": tau, "lif_dt": 1.0}


def _build_temporal_model(task, batch):
    from snntorch import surrogate
    from mnn_torch.models import TemporalMCSNN, TemporalMSNN
    if task == "seq-MNIST":
        return TemporalMSNN(28, 64, 10, beta=.95,
                            memristive_config=_temporal_config()).to(DEVICE_RESOLVED)
    if task == "SHD":
        return TemporalMSNN(700, 64, 20, beta=.95,
                            memristive_config=_temporal_config()).to(DEVICE_RESOLVED)
    shape = (2, 34, 34) if task == "N-MNIST" else (2, 32, 32)
    outputs = 10 if task == "N-MNIST" else 11
    return TemporalMCSNN(
        beta=.95, spike_grad=surrogate.fast_sigmoid(slope=25), num_steps=6,
        batch_size=batch, num_kernels=3, num_conv1=4, num_conv2=8,
        max_pooling=2, num_outputs=outputs, input_shape=shape,
        memristive_config={**_temporal_config(), "conv_ideal": True}).to(DEVICE_RESOLVED)


def _move(value):
    return value.to(DEVICE_RESOLVED, non_blocking=(DEVICE_RESOLVED == "cuda"))


def _temporal_forward(model, task, samples, gap=0):
    if task in {"seq-MNIST", "SHD"}:
        stream = samples.permute(1, 0, 2)
    else:
        stream = samples.permute(1, 0, 2, 3, 4)
    if gap:
        stream = torch.cat([stream, torch.zeros((gap, *stream.shape[1:]),
                                                device=stream.device)])
    return model(stream)[0]


def _set_model_tau(model, tau):
    from mnn_torch.models import beta_from_tau_leak
    beta = beta_from_tau_leak(tau, 1.0)
    for name in ("lif1", "lif2", "lif3"):
        if hasattr(model, name):
            lif = getattr(model, name)
            if torch.is_tensor(lif.beta):
                with torch.no_grad(): lif.beta.fill_(beta)
            else:
                lif.beta = beta


def _train_temporal_task(task, data, seed, budget):
    import torch.nn as nn
    torch.manual_seed(seed); np.random.seed(seed)
    train, test = _loaders(data, seed, budget["batch"])
    model = _build_temporal_model(task, budget["batch"])
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    epochs = budget["dense_epochs"] if task == "seq-MNIST" else budget["event_epochs"]
    for _ in range(epochs):
        model.train()
        for samples, labels in train:
            samples, labels = _move(samples), _move(labels)
            spikes = _temporal_forward(model, task, samples)
            loss = loss_fn(spikes.sum(0), labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    return model, test


def _evaluate_temporal(model, task, loader, tau=30.0, gap=0):
    _set_model_tau(model, tau); model.eval(); correct = total = 0
    with torch.no_grad():
        for samples, labels in loader:
            samples, labels = _move(samples), _move(labels)
            spikes = _temporal_forward(model, task, samples, gap)
            correct += (spikes.sum(0).argmax(1) == labels).sum().item()
            total += labels.shape[0]
    return 100.0 * correct / total


def _store_recall_trials(seeds):
    from mnn_torch.models import AdaptiveLeaky, DeviceLeaky
    delays = [2, 5, 10, 20, 40, 60]
    arms = ["fast LIF", "device 1.3", "ALIF 1.3", "device 8", "ALIF 8",
            "device 20", "ALIF 20"]
    panels = {
        "ideal": {"gain": 1.0, "noise": .005, "fault": 0.0},
        "device healthy": {"gain": .8, "noise": .045, "fault": 0.0},
        "device faulty": {"gain": .8, "noise": .055, "fault": .3},
    }
    result = {}
    for panel, cfg in panels.items():
        acc = {arm: [] for arm in arms}
        for seed in seeds:
            row = {arm: [] for arm in arms}
            for delay in delays:
                for arm in arms:
                    generator = torch.Generator().manual_seed(seed * 10000 + delay * 101 + arms.index(arm))
                    n = 256; labels = torch.randint(0, 2, (n,), generator=generator)
                    cue = torch.nn.functional.one_hot(labels, 2).float()
                    gain = cfg["gain"] * (1 + .1 * torch.randn((n, 1), generator=generator))
                    if cfg["fault"]:
                        gain *= (torch.rand((n, 1), generator=generator) > cfg["fault"])
                    if arm == "fast LIF" or arm.startswith("device"):
                        tau = .5 if arm == "fast LIF" else float(arm.split()[1])
                        neuron = DeviceLeaky(tau_leak=tau, dt=1, threshold=100)
                        mem = neuron.init_leaky(); _, mem = neuron(cue * gain, mem)
                        for _ in range(delay):
                            drive = cfg["noise"] * torch.randn((n, 2), generator=generator)
                            _, mem = neuron(drive, mem)
                        state = mem
                    else:
                        tau = float(arm.split()[1])
                        neuron = AdaptiveLeaky(beta=.2, tau_a=tau, dt=1,
                                               gamma=1.6, threshold=.5)
                        mem, adaptation = neuron.init_alif((n, 2))
                        _, mem, adaptation = neuron(cue * gain, mem, adaptation)
                        for _ in range(delay):
                            _, mem, adaptation = neuron(torch.zeros((n, 2)), mem, adaptation)
                        state = adaptation
                    state = state + cfg["noise"] * torch.randn((n, 2), generator=generator)
                    row[arm].append(100.0 * (state.argmax(1) == labels).float().mean().item())
            for arm in arms: acc[arm].append(row[arm])
        result[panel] = {"delays": delays, "chance": 50.0, "criterion": 75.0,
                         "acc": acc}
    return result


def run_live_temporal(profile):
    import time
    tz, mz, fixture_metadata = _load_reduced_fixtures()
    budget = temporal_budget(profile); started = time.time()
    tasks = ["seq-MNIST", "N-MNIST", "SHD", "DVS128 Gesture"]
    capability = {task: [] for task in tasks}
    trained = {}; loaders = {}; task_data = {
        task: _task_data(task, tz, mz, budget) for task in tasks}
    for task in tasks:
        for seed in budget["seeds"]:
            model, test = _train_temporal_task(task, task_data[task], seed, budget)
            capability[task].append(_evaluate_temporal(model, task, test))
            if task in {"seq-MNIST", "N-MNIST"}:
                trained[(task, seed)] = model; loaders[(task, seed)] = test
    retention = {}
    for task, modality, gaps in (
        ("seq-MNIST", "dense", budget["dense_gaps"]),
        ("N-MNIST", "conv", budget["conv_gaps"]),
    ):
        acc = {}
        for label, tau in (("1.3", 1.3), ("3", 3.0), ("8", 8.0),
                           ("20", 20.0), ("30", 30.0), ("no memory", .02)):
            acc[label] = [[_evaluate_temporal(trained[(task, seed)], task,
                           loaders[(task, seed)], tau=tau, gap=gap) for gap in gaps]
                          for seed in budget["seeds"]]
        retention[modality] = {"gaps": gaps, "chance": 10.0,
                               "criterion": 30.0 if modality == "dense" else 35.0,
                               "acc": acc}
    return {"metadata": {"profile": profile, "seeds": budget["seeds"],
            "device": DEVICE_RESOLVED, "budget": budget,
            "fixture_metadata": fixture_metadata}, "retention": retention,
            "store_recall": _store_recall_trials(budget["seeds"]),
            "capability": capability, "wall_seconds": time.time() - started}


seed_archive = load_seed_archive() if (USE_TEMPORAL_ARCHIVE or RUN_PROFILE == "publication") else None
if USE_TEMPORAL_ARCHIVE and seed_archive is None:
    raise FileNotFoundError(f"MNN_USE_TEMPORAL_ARCHIVE=1 but no compatible archive exists at {SEED_ARCHIVE}")
if RUN_PROFILE in {"reduced", "smoke"}:
    TEMPORAL_RESULTS = run_live_temporal(RUN_PROFILE)
    temporal_source = TEMPORAL_RESULTS; temporal_seeds = TEMPORAL_RESULTS["metadata"]["seeds"]
    temporal_provenance = "live-reduced"; temporal_claim = "reduced-validation"
elif seed_archive is not None:
    TEMPORAL_RESULTS = seed_archive; temporal_source = SEED_ARCHIVE
    temporal_seeds = seed_archive["metadata"]["seeds"]
    temporal_provenance = "live-exact"; temporal_claim = "claimable-runner-archive"
else:
    TEMPORAL_RESULTS = None; temporal_source = None; temporal_seeds = []
    temporal_provenance = "reference-only"; temporal_claim = "not-regenerated"


def _retention_panel(ax, panel, title):
    gaps = np.asarray(panel["gaps"], float); cmap = plt.cm.viridis
    keys = [key for key in panel["acc"] if key != "no memory"]
    for index, key in enumerate(keys):
        rows = np.asarray(panel["acc"][key], float)
        ci = np.array([boot_ci(rows[:, j], seed=j) for j in range(len(gaps))])
        colour = cmap(index / max(1, len(keys) - 1))
        ax.plot(gaps, rows.mean(0), "-o", ms=3, color=colour,
                label=fr"$\tau={key}$")
        ax.fill_between(gaps, ci[:, 0], ci[:, 1], color=colour, alpha=.12)
    no_memory = np.asarray(panel["acc"]["no memory"], float)
    ax.plot(gaps, no_memory.mean(0), "--", color=RED, lw=1.4, label="no memory")
    ax.axhline(panel["chance"], ls=":", color=GREY)
    ax.axhline(panel["criterion"], ls="--", color="0.4", lw=.8)
    ax.set_xlabel("delay $D$ (blank steps)", fontsize=12)
    ax.set_ylabel("test accuracy (%)", fontsize=12); ax.set_title(title, fontsize=13)
    ax.tick_params(labelsize=10.5)


if TEMPORAL_RESULTS is None:
    show_reference_only("fig_temporal_results.png", "publication archive not supplied")
else:
    n = len(temporal_seeds); fig, (axd, axc) = plt.subplots(1, 2, figsize=(11, 4.4))
    suffix = "reduced seeds" if temporal_provenance == "live-reduced" else "seeds"
    _retention_panel(axd, TEMPORAL_RESULTS["retention"]["dense"],
                     f"(a) seq-MNIST (dense, {n} {suffix})")
    _retention_panel(axc, TEMPORAL_RESULTS["retention"]["conv"],
                     f"(b) N-MNIST (conv, {n} {suffix})")
    handles, labels = axc.get_legend_handles_labels()
    fig.legend(handles, labels, fontsize=9.5, ncol=3, frameon=False,
               loc="lower center", bbox_to_anchor=(.5, -.02), columnspacing=1.6,
               handlelength=1.8)
    fig.tight_layout(rect=(0, .11, 1, 1))
    finish(fig, "fig_temporal_results.png", temporal_source,
           REFERENCE_DIR / "fig_temporal_results.png", seeds=temporal_seeds,
           dpi=200, provenance_class=temporal_provenance,
           claim_status=temporal_claim)

{"profile": RUN_PROFILE, "source": "live reduced trials" if TEMPORAL_RESULTS and temporal_provenance == "live-reduced" else "optional archive/reference",
 "wall_seconds": TEMPORAL_RESULTS.get("wall_seconds") if TEMPORAL_RESULTS else None,
 "capability": TEMPORAL_RESULTS.get("capability") if TEMPORAL_RESULTS else None}


## Live store–recall ladder

Each point is computed from 256 binary-cue trials per seed. The device arms use the package's `DeviceLeaky` neuron; ALIF controls use `AdaptiveLeaky`; the fast-LIF arm uses the same device-grounded class with an effectively short retention. Healthy and faulty panels apply documented gain variability, read noise, and a fixed cue-line fault probability. No accuracy curve is hand-shaped.

In [ ]:
INDIGO = "#5b6fb0"
SR_STYLES = {
    "fast LIF": dict(ls="--", marker="s", color=RED, label="fast LIF (no memory)"),
    "device 1.3": dict(ls="-", marker="o", color=TEAL, label=r"device $\tau_{\mathrm{leak}}=1.3$ (meas.)"),
    "ALIF 1.3": dict(ls="--", marker="^", color=TEAL, label=r"ALIF $\tau_a=1.3$"),
    "device 8": dict(ls="-", marker="o", color=INDIGO, label=r"device $\tau_{\mathrm{leak}}=8$ (meas.)"),
    "ALIF 8": dict(ls="--", marker="^", color=INDIGO, label=r"ALIF $\tau_a=8$"),
    "device 20": dict(ls="-", marker="o", color=AMBER, label=r"device $\tau_{\mathrm{leak}}=20$ (target)"),
    "ALIF 20": dict(ls="--", marker="^", color=AMBER, label=r"ALIF $\tau_a=20$"),
}


def _store_panel(ax, panel, title):
    delays = np.asarray(panel["delays"], float)
    for key, style in SR_STYLES.items():
        rows = np.asarray(panel["acc"][key], float)
        ax.plot(delays, rows.mean(0), ms=3, lw=1.4, **style)
        if key.startswith("device"):
            ci = np.array([boot_ci(rows[:, j], seed=j) for j in range(len(delays))])
            ax.fill_between(delays, ci[:, 0], ci[:, 1],
                            color=style["color"], alpha=.10)
    ax.axhline(panel["chance"], ls=":", color=GREY)
    ax.axhline(panel["criterion"], ls="--", color="0.4", lw=.8)
    ax.grid(True, which="major", ls="-", lw=.5, color=".88", zorder=0)
    ax.set_axisbelow(True); ax.set_ylabel("recall accuracy (%)", fontsize=12)
    ax.set_title(title, fontsize=13, loc="left"); ax.set_ylim(45, 108)
    ax.tick_params(labelsize=10.5)


if TEMPORAL_RESULTS is None:
    show_reference_only("fig_storerecall.png", "publication archive not supplied")
else:
    rungs = [("ideal", "(a) ideal synapse"),
             ("device healthy", "(b) device synapse, healthy"),
             ("device faulty", "(c) device synapse, faulty")]
    fig, axes = plt.subplots(3, 1, figsize=(6.6, 10.5), sharex=True,
                             constrained_layout=True)
    for ax, (key, title) in zip(axes, rungs):
        _store_panel(ax, TEMPORAL_RESULTS["store_recall"][key], title)
        ax.set_xlabel("")
    axes[-1].set_xlabel("store--recall delay $D$ (blank steps)", fontsize=12)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, fontsize=10.5, frameon=False, ncol=3,
               loc="outside lower center")
    finish(fig, "fig_storerecall.png", temporal_source,
           REFERENCE_DIR / "fig_storerecall.png", seeds=temporal_seeds,
           dpi=200, pad_inches=.02, provenance_class=temporal_provenance,
           claim_status=temporal_claim)


## Live reduced benchmark capability

All four bars come from trained temporal networks evaluated on held-out samples. N-MNIST, SHD, and DVS Gesture use deterministic reduced repartitions of genuine official test distributions because packaging their multi-gigabyte training distributions would violate the lightweight release goal. The title and provenance make this nonstandard validation split explicit; exact manuscript benchmark scores require the optional validated full-sweep archive.

In [ ]:
names = ["seq-MNIST\n(dense)", "N-MNIST\n(conv)", "SHD\n(dense)",
         "DVS Gesture\n(conv)"]
keys = ["seq-MNIST", "N-MNIST", "SHD", "DVS128 Gesture"]
if TEMPORAL_RESULTS is None:
    show_reference_only("fig_capability.png", "publication archive not supplied")
else:
    rows = [np.asarray(TEMPORAL_RESULTS["capability"][key], float) for key in keys]
    accs = np.array([row.mean() for row in rows])
    ci = np.array([boot_ci(row, seed=1) for row in rows])
    chances = [10, 10, 5, 100/11]; x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(6, 4))
    err = np.vstack((accs-ci[:, 0], ci[:, 1]-accs))
    ax.bar(x, accs, color=[BLUE, TEAL, BLUE, TEAL], edgecolor=INK,
           linewidth=.6, yerr=err, capsize=4,
           error_kw=dict(ecolor=INK, lw=1))
    for xi, chance in zip(x, chances):
        ax.plot([xi-.4, xi+.4], [chance, chance], ls=":", color=GREY, lw=1.2)
    for xi, value in zip(x, accs):
        ax.text(xi, value+2, f"{value:.1f}", ha="center", fontsize=8, color=INK)
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=8.5)
    ax.set_ylabel("test accuracy (%)", fontsize=9); ax.set_ylim(0, 100)
    qualifier = "reduced sample validation" if temporal_provenance == "live-reduced" else "full-sweep archive"
    ax.set_title(f"Benchmark capability ({qualifier}; dotted = chance)", fontsize=10)
    fig.tight_layout()
    finish(fig, "fig_capability.png", temporal_source,
           REFERENCE_DIR / "fig_capability.png", seeds=temporal_seeds,
           dpi=200, bbox_inches=None, provenance_class=temporal_provenance,
           claim_status=temporal_claim)


## Reproduction report

In [ ]:
assert len(FIGURE_REPORT) == len([x for x in MANIFEST["figures"] if x["notebook"] == Path(globals().get("NOTEBOOK_NAME", "")).name])
FIGURE_REPORT